# Cleaning:
1. Remove admission_ids which have both icu_mortality and mortality_after_discharge as True
3. [Done before gridding which is more appropriate] Remove row in each admission which are very close (<=5 hrs) to discharge. This is to avoid data leakage on discharge decision. Check if this can be done at 3_organizing pipeline.
4. Add A, D
5. Add Y (All-cause mortality - in-icu mortality or mortality after discharge)

In [1]:
import pandas as pd
import numpy as np

In [2]:
sf_mimicdata = pd.read_parquet('../Dataset/mimic-iv/from_pipeline/ltm_clipped_grided_standard.parquet')

In [3]:
sf_mimicdata

,stay_id,k,grid_end,temperature__mean__last_12h,heart_rate__mean__last_12h,arterial_blood_pressure_mean__mean__last_12h,fluid_out_urine__mean__last_12h,pco2_arterial__mean__last_12h,respiratory_rate_measured__mean__last_12h,o2_saturation__mean__last_12h,...,raw_weight,unit_type,origin,los,intime,outtime,death_time_from_intime,icu_mortality,death_abs_time,icu_mortality_derived
0,mimic4-30000153,1,0 days 12:00:00,37.625000,109.076920,84.380959,82.222221,44.0,17.230770,97.923080,...,73.0000,Trauma,Emergency,1 days 15:17:10,2174-09-29 12:09:00,2174-10-01 03:26:10,NaT,0,NaT,False
1,mimic4-30000153,2,1 days 00:00:00,37.333332,104.416664,91.846153,55.000000,NaN,12.500000,95.250000,...,73.0000,Trauma,Emergency,1 days 15:17:10,2174-09-29 12:09:00,2174-10-01 03:26:10,NaT,0,NaT,False
2,mimic4-30000213,1,0 days 12:00:00,36.911114,78.384613,78.846153,324.166656,41.5,21.692308,98.846153,...,84.7000,Medical,Other,1 days 15:14:48,2162-06-21 05:38:00,2162-06-22 20:52:48,1363 days 18:22:00,0,2166-03-16,False
3,mimic4-30000213,2,1 days 00:00:00,37.166668,85.250000,79.416664,230.000000,NaN,16.250000,99.083336,...,84.7000,Medical,Other,1 days 15:14:48,2162-06-21 05:38:00,2162-06-22 20:52:48,1363 days 18:22:00,0,2166-03-16,False
4,mimic4-30000484,1,0 days 12:00:00,35.629631,89.166664,64.750000,114.000000,59.0,15.000000,99.750000,...,68.4178,Medical,Emergency,2 days 11:29:35.999999,2136-01-14 17:23:32,2136-01-17 04:53:08,37 days 06:36:28,0,2136-02-21,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574709,mimic4-39999858,3,1 days 12:00:00,36.777782,66.785713,77.250000,NaN,NaN,23.357143,93.071426,...,123.8000,Medical-Surgical,Direct Admit,3 days 21:35:50,2167-04-26 12:36:00,2167-04-30 10:11:50,NaT,0,NaT,False
574710,mimic4-39999858,4,2 days 00:00:00,36.944447,64.833336,77.250000,NaN,NaN,21.500000,95.166664,...,123.8000,Medical-Surgical,Direct Admit,3 days 21:35:50,2167-04-26 12:36:00,2167-04-30 10:11:50,NaT,0,NaT,False
574711,mimic4-39999858,5,2 days 12:00:00,36.902779,64.250000,81.818184,NaN,NaN,23.916666,92.550003,...,123.8000,Medical-Surgical,Direct Admit,3 days 21:35:50,2167-04-26 12:36:00,2167-04-30 10:11:50,NaT,0,NaT,False
574712,mimic4-39999858,6,3 days 00:00:00,36.870373,68.166664,71.166664,NaN,NaN,24.500000,93.250000,...,123.8000,Medical-Surgical,Direct Admit,3 days 21:35:50,2167-04-26 12:36:00,2167-04-30 10:11:50,NaT,0,NaT,False


In [4]:
sf_mimicdata["timestep"] = (
    sf_mimicdata.sort_values(["stay_id", "grid_end"])
              .groupby("stay_id")
              .cumcount()
)

In [5]:
sf_mimicdata = sf_mimicdata.rename(columns={"timestep": "t"})

In [6]:
sf_mimicdata.columns

Index(['stay_id', 'k', 'grid_end', 'temperature__mean__last_12h',
       'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'fio2__last__last_12h',
       'vent_mode__last__last_12h', 'vent_mode__hours_since_last__last_12h',
       'n_obs_in_window', 'subject_id', 'raw_age', 'sex', 'raw_height',
       'raw_weight', 'unit_type', 'origin', 'los', 'intime', 'outtime',
       'death_time_from_intime', 'icu_mortality', 'death_abs_time',
       'icu_mortality_derived', 't'],
    

In [7]:
sf_mimicdata = sf_mimicdata.sort_values(by=['stay_id', 't'])

In [8]:
sf_mimicdata.stay_id.nunique()

82540

In [9]:
nan_columns = sf_mimicdata.columns[sf_mimicdata.isna().all()].tolist()
nan_columns

[]

In [10]:
#mimicdata = mimicdata.drop(columns=nan_columns)

In [11]:
sf_mimicdata.columns

Index(['stay_id', 'k', 'grid_end', 'temperature__mean__last_12h',
       'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'fio2__last__last_12h',
       'vent_mode__last__last_12h', 'vent_mode__hours_since_last__last_12h',
       'n_obs_in_window', 'subject_id', 'raw_age', 'sex', 'raw_height',
       'raw_weight', 'unit_type', 'origin', 'los', 'intime', 'outtime',
       'death_time_from_intime', 'icu_mortality', 'death_abs_time',
       'icu_mortality_derived', 't'],
    

In [12]:
sf_mimicdata.head(10)

,stay_id,k,grid_end,temperature__mean__last_12h,heart_rate__mean__last_12h,arterial_blood_pressure_mean__mean__last_12h,fluid_out_urine__mean__last_12h,pco2_arterial__mean__last_12h,respiratory_rate_measured__mean__last_12h,o2_saturation__mean__last_12h,...,unit_type,origin,los,intime,outtime,death_time_from_intime,icu_mortality,death_abs_time,icu_mortality_derived,t
0,mimic4-30000153,1,0 days 12:00:00,37.625000,109.076920,84.380959,82.222221,44.0,17.230770,97.923080,...,Trauma,Emergency,1 days 15:17:10,2174-09-29 12:09:00,2174-10-01 03:26:10,NaT,0,NaT,False,0
1,mimic4-30000153,2,1 days 00:00:00,37.333332,104.416664,91.846153,55.000000,NaN,12.500000,95.250000,...,Trauma,Emergency,1 days 15:17:10,2174-09-29 12:09:00,2174-10-01 03:26:10,NaT,0,NaT,False,1
2,mimic4-30000213,1,0 days 12:00:00,36.911114,78.384613,78.846153,324.166656,41.5,21.692308,98.846153,...,Medical,Other,1 days 15:14:48,2162-06-21 05:38:00,2162-06-22 20:52:48,1363 days 18:22:00,0,2166-03-16,False,0
3,mimic4-30000213,2,1 days 00:00:00,37.166668,85.250000,79.416664,230.000000,NaN,16.250000,99.083336,...,Medical,Other,1 days 15:14:48,2162-06-21 05:38:00,2162-06-22 20:52:48,1363 days 18:22:00,0,2166-03-16,False,1
4,mimic4-30000484,1,0 days 12:00:00,35.629631,89.166664,64.750000,114.000000,59.0,15.000000,99.750000,...,Medical,Emergency,2 days 11:29:35.999999,2136-01-14 17:23:32,2136-01-17 04:53:08,37 days 06:36:28,0,2136-02-21,False,0
5,mimic4-30000484,2,1 days 00:00:00,36.125004,90.750000,69.487175,71.111115,NaN,14.416667,99.833336,...,Medical,Emergency,2 days 11:29:35.999999,2136-01-14 17:23:32,2136-01-17 04:53:08,37 days 06:36:28,0,2136-02-21,False,1
6,mimic4-30000484,3,1 days 12:00:00,36.759262,94.833336,79.416664,86.666664,NaN,19.750000,100.000000,...,Medical,Emergency,2 days 11:29:35.999999,2136-01-14 17:23:32,2136-01-17 04:53:08,37 days 06:36:28,0,2136-02-21,False,2
7,mimic4-30000484,4,2 days 00:00:00,36.222225,91.500000,77.000000,50.000000,NaN,18.750000,100.000000,...,Medical,Emergency,2 days 11:29:35.999999,2136-01-14 17:23:32,2136-01-17 04:53:08,37 days 06:36:28,0,2136-02-21,False,3
8,mimic4-30000646,1,0 days 12:00:00,37.755558,89.606064,64.551727,NaN,30.0,28.878788,95.794121,...,Cardiac,Emergency,4 days 16:44:26,2194-04-29 01:39:22,2194-05-03 18:23:48,6 days 22:20:38,1,2194-05-06,False,0
9,mimic4-30000646,2,1 days 00:00:00,36.740742,82.384613,64.800003,NaN,NaN,21.500000,95.307693,...,Cardiac,Emergency,4 days 16:44:26,2194-04-29 01:39:22,2194-05-03 18:23:48,6 days 22:20:38,1,2194-05-06,False,1


In [13]:
sf_mimicdata.iloc[0]

stay_id                                                      mimic4-30000153
k                                                                          1
grid_end                                                     0 days 12:00:00
temperature__mean__last_12h                                           37.625
heart_rate__mean__last_12h                                         109.07692
arterial_blood_pressure_mean__mean__last_12h                       84.380959
fluid_out_urine__mean__last_12h                                    82.222221
pco2_arterial__mean__last_12h                                           44.0
respiratory_rate_measured__mean__last_12h                           17.23077
o2_saturation__mean__last_12h                                       97.92308
po2_arterial__mean__last_12h                                           233.0
bicarbonate_arterial__last__last_12h                                    19.0
activated_partial_thromboplastin_time__last__last_12h              25.299999

# Check if there are any stays with both icu_mortality=True and mortality_after_discharge=True

In [14]:
sf_mimicdata = sf_mimicdata.rename(columns={"icu_mortality": "icu_mortality_flag", "icu_mortality_derived": "icu_mortality"})


In [15]:
sf_mimicdata.columns

Index(['stay_id', 'k', 'grid_end', 'temperature__mean__last_12h',
       'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'fio2__last__last_12h',
       'vent_mode__last__last_12h', 'vent_mode__hours_since_last__last_12h',
       'n_obs_in_window', 'subject_id', 'raw_age', 'sex', 'raw_height',
       'raw_weight', 'unit_type', 'origin', 'los', 'intime', 'outtime',
       'death_time_from_intime', 'icu_mortality_flag', 'death_abs_time',
       'icu_mortality', 't'],
      d

In [16]:
#ratio of mortality
last_rows = sf_mimicdata.groupby("stay_id").tail(1)
(last_rows["icu_mortality"] == 1).mean()

np.float64(0.07397625393748486)

In [17]:
sf_mimicdata.stay_id.nunique()

82540

In [18]:
sf_mimicdata = sf_mimicdata.sort_values(by=['stay_id', 't'])

In [19]:
sf_mimicdata.stay_id.nunique()

82540

## Mortality within X days after discharge

In [20]:
# Ensure correct dtypes
sf_mimicdata["intime"]  = pd.to_datetime(sf_mimicdata["intime"], errors="coerce")
sf_mimicdata["outtime"] = pd.to_datetime(sf_mimicdata["outtime"], errors="coerce")

# death_time_from_intime must be timedelta
sf_mimicdata["death_time_from_intime"] = pd.to_timedelta(
    sf_mimicdata["death_time_from_intime"], errors="coerce"
)

# Absolute death time = intime + death offset
sf_mimicdata["death_abs_time"] = sf_mimicdata["intime"] + sf_mimicdata["death_time_from_intime"]

# Post-discharge mortality [outtime < death_abs_time ≤ (intime+12h)+30d]
HORIZON = pd.Timedelta(days=30)
sf_mimicdata["mortality_after_discharge_30d"] = (
    sf_mimicdata["death_abs_time"].notna()
    & (sf_mimicdata["death_abs_time"] > sf_mimicdata["outtime"])
    & (sf_mimicdata["death_abs_time"] <=
       (sf_mimicdata["intime"] + pd.Timedelta(hours=12) + HORIZON))
).astype(int)

In [21]:
#ratio of mortality
last_rows = sf_mimicdata.groupby("stay_id").tail(1)
(last_rows["mortality_after_discharge_30d"] == 1).mean()

np.float64(0.07483801168902948)

# Add A, Y and D

In [22]:
# Defaults: ongoing rows (A=0, D=0) -> Z and Y is missing. Therefore, nan
sf_mimicdata.loc[:, 'A'] = 0
sf_mimicdata.loc[:, 'D'] = 0
sf_mimicdata.loc[:, 'Z'] = np.nan
sf_mimicdata.loc[:, 'Y'] = np.nan

In [23]:
# Ensure correct ordering within stay
sf_mimicdata = sf_mimicdata.sort_values(["stay_id", "t"]).copy()

# Fill A
# Update "A" to 0 if "icu_mortality" is True, and A=1 in other case (~ icu_mortality) at the last row of each 'stay_id'
sf_mimicdata.loc[sf_mimicdata.groupby('stay_id').tail(1).index, 'A'] = sf_mimicdata.groupby('stay_id').tail(1)['icu_mortality'].apply(lambda x: 0 if x else 1)

# indices of terminal (last) row per stay
last_idx = sf_mimicdata.groupby("stay_id", sort=False).tail(1).index

# set D on terminal row from icu_mortality (True/False -> 1/0)
sf_mimicdata.loc[last_idx, "D"] = sf_mimicdata.loc[last_idx, "icu_mortality"].astype(int)

# terminal reason 1: in-ICU death (A=0, D=1) -> Y = 1
death_mask = sf_mimicdata.loc[last_idx, "D"].eq(1)
sf_mimicdata.loc[last_idx[death_mask], "Y"] = 1

# Fill Z
disch_mask = sf_mimicdata.loc[last_idx, "A"].eq(1) & sf_mimicdata.loc[last_idx, "D"].eq(0)
sf_mimicdata.loc[last_idx[disch_mask], "Z"] = sf_mimicdata.loc[last_idx[disch_mask], "mortality_after_discharge_30d"]
sf_mimicdata.loc[last_idx[disch_mask], "D"] = np.nan # To declare in-icu death as NA for discharged stays.

# Y will be filled below after truncation of stays for 90 days with Z

# Limit the study to 30 days since admission. 
# For stay_ids which have LOS lesser than 30days since admission, keep them as is.
# For stays which are more than 30 days since admission, chop the rows to have LOS at 30 days only and declare those stays to have D=0, Z=0, Y=0 at the last row

In [24]:
def truncate_followup_30d_t(
    df,
    id_col="stay_id",
    t_col="t",
    intime_col="intime",
    outcome_cols=("A", "D", "Z", "Y"),
    bin_hours=12,
    death_td_col="death_time_from_intime"
):
    """
    Truncate longitudinal ICU data at K = 30 days since (intime + 12h) on a bin_hours grid.

    For stays exceeding K:
      - keep rows with t <= t_cutoff
      - at t == t_cutoff: set D=0, Y=0 (admin censoring)
      - at t == t_cutoff: set Z=NaN (post-discharge outcome undefined because not discharged by K)

    Parameters
    ----------
    df : pandas.DataFrame
        Long-format data with one row per (stay_id, t).
    id_col : str
        Stay identifier column.
    t_col : str
        Discrete time index column.
    intime_col : str
        ICU admission timestamp column.
    outcome_cols : tuple
        Outcome columns to zero at administrative cutoff.
    bin_hours : int
        Width of each discrete time interval (default 12h).

    Returns
    -------
    pandas.DataFrame
        Truncated dataframe.
    """

    df = df.copy()

    # Ensure proper types
    df[intime_col] = pd.to_datetime(df[intime_col], errors="coerce")

    # Ensure outcomes exist
    for c in outcome_cols:
        if c not in df.columns:
            df[c] = np.nan

    # Sort for consistency
    df = df.sort_values([id_col, t_col]).reset_index(drop=True)

    # --- cutoff at 30d since (intime + bin_hours) ---
    cutoff_time = df[intime_col] + pd.DateOffset(days=30) + pd.Timedelta(hours=bin_hours)

    # Convert cutoff_time → max allowed t
    delta_hours = (cutoff_time - df[intime_col]).dt.total_seconds() / 3600.0
    t_cutoff = np.floor(delta_hours / bin_hours).astype(int) - 1  # -> xx when bin_hours=12

    # Constant per stay
    df["_t_cutoff"] = df.groupby(id_col)[t_col].transform(
        lambda x: t_cutoff.loc[x.index].min()
    )

    # Identify long stays
    max_t = df.groupby(id_col)[t_col].transform("max")
    df["_long"] = max_t > df["_t_cutoff"]

    # Keep rows within cutoff
    kept = df[(~df["_long"]) | (df[t_col] <= df["_t_cutoff"])].copy()

    # For truncated stays, handle outcomes at final kept row
    mask_last = kept["_long"] & (kept[t_col] == kept["_t_cutoff"])

    # Admin censoring for time-local outcomes (D, Y, etc.), but NOT Z
    cols_to_zero = [c for c in outcome_cols if c not in ("Z",)]
    if cols_to_zero:
        kept.loc[mask_last, cols_to_zero] = 0

    # Post-discharge outcome undefined if not discharged by K
    if "Z" in outcome_cols:
        kept.loc[mask_last, "Z"] = np.nan
        kept.loc[mask_last, "Y"] = 0

    # Cleanup
    kept = kept.drop(columns=["_t_cutoff", "_long"], errors="ignore")
    kept = kept.sort_values([id_col, t_col]).reset_index(drop=True)

    return kept

In [25]:
sf_mimicdata_90d = truncate_followup_30d_t(sf_mimicdata)

# terminal row per stay *after truncation*
last_idx_90d = sf_mimicdata_90d.groupby("stay_id", sort=False).tail(1).index

# 1) In-ICU death at terminal row: (A=0, D=1) -> Y=1, Z must be NaN
icu_death = sf_mimicdata_90d.loc[last_idx_90d, "A"].eq(0) & sf_mimicdata_90d.loc[last_idx_90d, "D"].eq(1)
sf_mimicdata_90d.loc[last_idx_90d[icu_death], "Y"] = 1
sf_mimicdata_90d.loc[last_idx_90d[icu_death], "Z"] = np.nan

# 2) Discharged terminal row: (A=1, D=NA) -> Y = Z (0/1)
discharged = sf_mimicdata_90d.loc[last_idx_90d, "A"].eq(1) & sf_mimicdata_90d.loc[last_idx_90d, "D"].isna()
sf_mimicdata_90d.loc[last_idx_90d[discharged], "Y"] = sf_mimicdata_90d.loc[last_idx_90d[discharged], "Z"]

# 3) Admin censored in ICU at K: (A=0, D=0, Z=NaN) -> Y=0
censored = sf_mimicdata_90d.loc[last_idx_90d, "A"].eq(0) & sf_mimicdata_90d.loc[last_idx_90d, "D"].eq(0)
sf_mimicdata_90d.loc[last_idx_90d[censored], "Y"] = 0
sf_mimicdata_90d.loc[last_idx_90d[censored], "Z"] = np.nan

In [26]:
sf_mimicdata_90d.groupby('stay_id')['D'].max().mean()

np.float64(0.07245081552735867)

In [27]:
sf_mimicdata_90d.groupby('stay_id')['Z'].max().mean()

np.float64(0.08135714559656367)

In [28]:
sf_mimicdata_90d.groupby('stay_id')['Y'].max().mean()

np.float64(0.14728882721638817)

In [29]:
0.07376876911939845 + 0.14012706736587333

0.21389583648527177

In [30]:
sf_mimicdata_90d.groupby(['A', 'D', 'Z', 'Y'], dropna=False).size().reset_index(name='count')

,A,D,Z,Y,count
0,0,0.0,NaN,0.0,653
1,0,0.0,NaN,NaN,616042
2,0,1.0,NaN,1.0,6161
3,1,NaN,0.0,0.0,71859
4,1,NaN,1.0,1.0,6364


In [31]:
last_rows = (
    sf_mimicdata_90d
    .sort_values(['stay_id', 't'])
    .groupby('stay_id')
    .tail(1)
)

In [32]:
last_rows[['A', 'D', 'Z', 'Y']].drop_duplicates().reset_index(drop=True)

,A,D,Z,Y
0,1,NaN,0.0,0.0
1,1,NaN,1.0,1.0
2,0,1.0,NaN,1.0
3,0,0.0,NaN,0.0


In [33]:
sf_mimicdata_90d.to_parquet('../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet')

In [34]:
sf_mimicdata_90d.columns

Index(['stay_id', 'grid_end', 'vent_mode__hours_since_last__last_12h',
       'temperature__mean__last_12h', 'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'fio2__last__last_12h',
       'vent_mode__last__last_12h', 'subject_id', 'raw_age', 'sex',
       'raw_height', 'raw_weight', 'unit_type', 'origin', 'los', 'intime',
       'outtime', 'death_time_from_intime', 'icu_mortality_flag',
       'death_abs_time', 'icu_mortality', 't', 'mortality_after_discharge_30d',
      

# Cross-checking that A, Y and D are behaving well

In [35]:
sf_mimicdata_90d.loc[sf_mimicdata_90d.Z==1].stay_id.unique()

array(['mimic4-30000646', 'mimic4-30004232', 'mimic4-30007228', ...,
       'mimic4-39993425', 'mimic4-39995141', 'mimic4-39999168'],
      shape=(6364,), dtype=object)

In [36]:
sf_mimicdata_90d.loc[sf_mimicdata_90d.stay_id=="mimic4-30000484"]

,stay_id,grid_end,vent_mode__hours_since_last__last_12h,temperature__mean__last_12h,heart_rate__mean__last_12h,arterial_blood_pressure_mean__mean__last_12h,fluid_out_urine__mean__last_12h,pco2_arterial__mean__last_12h,respiratory_rate_measured__mean__last_12h,o2_saturation__mean__last_12h,...,death_time_from_intime,icu_mortality_flag,death_abs_time,icu_mortality,t,mortality_after_discharge_30d,A,D,Z,Y
8,mimic4-30000484,0 days 12:00:00,NaN,35.629631,89.166664,64.750000,114.000000,59.0,15.000000,99.750000,...,37 days 06:36:28,0,2136-02-21,False,0,0,0,0.0,NaN,NaN
9,mimic4-30000484,1 days 00:00:00,NaN,36.125000,90.750000,69.487183,71.111115,NaN,14.416667,99.833336,...,37 days 06:36:28,0,2136-02-21,False,1,0,0,0.0,NaN,NaN
10,mimic4-30000484,1 days 12:00:00,NaN,36.759258,94.833336,79.416664,86.666664,NaN,19.750000,100.000000,...,37 days 06:36:28,0,2136-02-21,False,2,0,0,0.0,NaN,NaN
11,mimic4-30000484,2 days 00:00:00,NaN,36.222225,91.500000,77.000000,50.000000,NaN,18.750000,100.000000,...,37 days 06:36:28,0,2136-02-21,False,3,0,0,0.0,NaN,NaN
12,mimic4-30000484,2 days 12:00:00,NaN,36.611115,93.000000,83.952377,58.750000,NaN,20.333334,99.666664,...,37 days 06:36:28,0,2136-02-21,False,4,0,1,NaN,0.0,0.0


In [37]:
sf_mimicdata_90d.t.max()

np.int64(60)

In [38]:
sf_mimicdata_90d.grid_end.max()

Timedelta('30 days 12:00:00')